# Compare TRF predictors for subject 01

This notebook runs the same subject through several registered TRF models and stores a small comparison table. The large TRF objects stay in the Eelbrain cache; the CSV here is the human-readable experiment log.

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys
import traceback

import numpy as np
import pandas as pd


def find_pipeline_dir(start=Path.cwd()):
    start = Path(start).resolve()
    candidates = [start, *start.parents, start / 'analysis' / 'trf_pipeline']
    for path in candidates:
        if (path / 'alice_eelbrain_main_experiment.py').exists():
            return path
    raise FileNotFoundError(f'Could not find alice_eelbrain_main_experiment.py from {start}')


PIPELINE_DIR = find_pipeline_dir()
PROJECT_ROOT = PIPELINE_DIR.parents[1]
RESULTS_DIR = PIPELINE_DIR / 'results'

if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from alice_eelbrain_main_experiment import TRF_OPTIONS, alice

print(f'Pipeline dir: {PIPELINE_DIR}')
print(f'Results dir: {RESULTS_DIR}')
print(f'TRF options: {TRF_OPTIONS}')

## Experiment setup

`word-onset` uses the `time` column in `*~word.pickle` and places a unit impulse at each word onset. The lexical predictors use boolean columns from the same word-level table.

In [ ]:
SUBJECT = '01'
RAW = '0.5-20'
EPOCH = 'story-segments'
INV = ''
ESTIMATOR = 'boosting'

MODEL_NAMES = [
    'gammatone-8',
    'acoustic-onset-8',
    'auditory-gammatone',
    'word-onset',
    'lexical-onset',
    'nonlexical-onset',
    'word-logfreq',
    'word-ngram',
    'word-rnn',
    'word-cfg',
    'gammatone-plus-word',
]

model_table = pd.DataFrame(
    [{'model': name, 'term': alice.models[name]} for name in MODEL_NAMES]
)
model_table

## Cache paths

Different model definitions should produce different cache files. If you rerun the exact same model and parameters, Eelbrain may reuse the same cached artifact.

In [ ]:
cache_rows = []
for model in MODEL_NAMES:
    try:
        cache_path = alice.load_trf(
            model,
            subject=SUBJECT,
            raw=RAW,
            epoch=EPOCH,
            inv=INV,
            estimator=ESTIMATOR,
            path_only=True,
            **TRF_OPTIONS,
        )
        cache_rows.append({
            'subject': SUBJECT,
            'model': model,
            'term': alice.models[model],
            'cache_path': str(cache_path),
            'cache_exists': Path(cache_path).exists(),
        })
    except Exception as error:
        cache_rows.append({
            'subject': SUBJECT,
            'model': model,
            'term': alice.models.get(model, ''),
            'cache_path': '',
            'cache_exists': False,
            'error': repr(error),
        })

cache_table = pd.DataFrame(cache_rows)
cache_table

## Run or load TRFs

This cell can take a while. Each row is fit or loaded independently so one failed model does not stop the whole comparison.

In [ ]:
def numeric_array(value):
    if hasattr(value, 'x'):
        value = value.x
    return np.asarray(value, dtype=float)


def summarize_result(result):
    r = numeric_array(result.r)
    h = numeric_array(result.h)
    return {
        'r_min': float(np.nanmin(r)),
        'r_max': float(np.nanmax(r)),
        'r_mean': float(np.nanmean(r)),
        'r_abs_mean': float(np.nanmean(np.abs(r))),
        'n_sensors': int(r.size),
        'h_shape': str(h.shape),
        'finite_r': bool(np.isfinite(r).all()),
        'finite_h': bool(np.isfinite(h).all()),
    }


RUN_MODELS = True

rows = []
for model in MODEL_NAMES:
    cache_path = alice.load_trf(
        model,
        subject=SUBJECT,
        raw=RAW,
        epoch=EPOCH,
        inv=INV,
        estimator=ESTIMATOR,
        path_only=True,
        **TRF_OPTIONS,
    )
    row = {
        'run_time': datetime.now().isoformat(timespec='seconds'),
        'subject': SUBJECT,
        'model': model,
        'term': alice.models[model],
        'estimator': ESTIMATOR,
        'raw': RAW,
        'epoch': EPOCH,
        'tstart': TRF_OPTIONS['tstart'],
        'tstop': TRF_OPTIONS['tstop'],
        'samplingrate': TRF_OPTIONS['samplingrate'],
        'filter_x': TRF_OPTIONS['filter_x'],
        'cache_path': str(cache_path),
        'cache_exists_before': Path(cache_path).exists(),
    }
    if RUN_MODELS:
        try:
            result = alice.load_trf(
                model,
                subject=SUBJECT,
                raw=RAW,
                epoch=EPOCH,
                inv=INV,
                estimator=ESTIMATOR,
                **TRF_OPTIONS,
            )
            row.update(summarize_result(result))
            row['status'] = 'ok'
        except Exception as error:
            row['status'] = 'error'
            row['error'] = repr(error)
            row['traceback'] = traceback.format_exc(limit=3)
    else:
        row['status'] = 'not_run'
    rows.append(row)
    print(f"{model}: {row['status']}")

comparison = pd.DataFrame(rows)
comparison

## Save comparison table

The CSV is intentionally small and reviewable. It is the table to use for comparing predictors; the cache path points back to the full TRF object.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
output_path = RESULTS_DIR / f'trf_predictor_comparison_sub-{SUBJECT}.csv'

if output_path.exists():
    previous = pd.read_csv(output_path)
    combined = pd.concat([previous, comparison], ignore_index=True, sort=False)
    dedupe_columns = ['subject', 'model', 'estimator', 'raw', 'epoch', 'tstart', 'tstop', 'samplingrate', 'filter_x']
    combined = combined.drop_duplicates(subset=dedupe_columns, keep='last')
else:
    combined = comparison

combined.to_csv(output_path, index=False)
print(f'Wrote {output_path}')
combined.sort_values('r_mean', ascending=False, na_position='last')